In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("yellow_tripdata_2021-01.csv.gz")

print("Dataset exists:", DATA_PATH.exists())

assert DATA_PATH.exists(), "Dataset not found. Check the data folder."

Dataset exists: True


In [3]:

from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("yellow_tripdata_2021-01.csv.gz")

print("Dataset exists:", DATA_PATH.exists())

df = pd.read_csv(DATA_PATH, nrows=100)

print("\nFirst 100 rows:")
print(df.head(100))

print("\nData types:")
print(df.dtypes)

print("\nShape:")
print(df.shape)


Dataset exists: True

First 100 rows:
    VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0          1  2021-01-01 00:30:10   2021-01-01 00:36:12                1   
1          1  2021-01-01 00:51:20   2021-01-01 00:52:19                1   
2          1  2021-01-01 00:43:30   2021-01-01 01:11:06                1   
3          1  2021-01-01 00:15:48   2021-01-01 00:31:01                0   
4          2  2021-01-01 00:31:49   2021-01-01 00:48:21                1   
..       ...                  ...                   ...              ...   
95         2  2021-01-01 00:12:41   2021-01-01 00:26:47                1   
96         2  2021-01-01 00:23:29   2021-01-01 00:35:03                2   
97         2  2021-01-01 00:46:17   2021-01-01 00:54:25                2   
98         2  2021-01-01 00:28:16   2021-01-01 00:51:44                1   
99         2  2021-01-01 00:42:35   2021-01-01 00:54:41                3   

    trip_distance  RatecodeID store_and_fwd_flag 

In [4]:

from pathlib import Path
import pandas as pd
import numpy as np

DATA_PATH = Path("yellow_tripdata_2021-01.csv.gz")
CHUNK_SIZE = 100_000

# Store profiling results without loading the full dataset
total_rows = 0
profile_data = {}

for chunk in pd.read_csv(DATA_PATH, chunksize=CHUNK_SIZE):
    total_rows += len(chunk)

    for col in chunk.columns:
        if col not in profile_data:
            profile_data[col] = {
                "dtype": str(chunk[col].dtype),
                "null_count": 0,
                "min": None,
                "max": None,
            }

        profile_data[col]["null_count"] += int(chunk[col].isna().sum())

        # Parse date columns for meaningful min/max values
        if col in ["tpep_pickup_datetime", "tpep_dropoff_datetime"]:
            values = pd.to_datetime(chunk[col], errors="coerce")
        else:
            values = chunk[col]

        valid = values.dropna()

        if not valid.empty:
            current_min = valid.min()
            current_max = valid.max()

            old_min = profile_data[col]["min"]
            old_max = profile_data[col]["max"]

            if old_min is None or current_min < old_min:
                profile_data[col]["min"] = current_min

            if old_max is None or current_max > old_max:
                profile_data[col]["max"] = current_max

# Build the final profiling table
profile = pd.DataFrame.from_dict(profile_data, orient="index")
profile.index.name = "column"
profile["null_pct"] = (
    profile["null_count"] / total_rows * 100
).round(2)

profile = profile[
    ["dtype", "null_count", "null_pct", "min", "max"]
]

print("Total rows:", total_rows)
print("\nDataset profiling:")
print(profile.to_string())


C:\Users\mohamed ahmed fathy\AppData\Local\Temp\ipykernel_30168\3753603482.py:12: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv(DATA_PATH, chunksize=CHUNK_SIZE):


Total rows: 1369765

Dataset profiling:
                         dtype  null_count  null_pct                  min                  max
column                                                                                        
VendorID                 int64       98352      7.18                    1                    2
tpep_pickup_datetime    object           0      0.00  2008-12-31 23:05:14  2021-02-22 16:52:16
tpep_dropoff_datetime   object           0      0.00  2008-12-31 23:07:22  2021-02-22 16:56:15
passenger_count          int64       98352      7.18                    0                    8
trip_distance          float64           0      0.00                  0.0            263163.28
RatecodeID               int64       98352      7.18                    1                   99
store_and_fwd_flag      object       98352      7.18                    N                    Y
PULocationID             int64           0      0.00                    1                  265
DOLocation

In [5]:

from pathlib import Path
import pandas as pd

DATA_PATH = Path("yellow_tripdata_2021-01.csv.gz")
CHUNK_SIZE = 100_000

issue_counts = {
    "pickup_outside_january_2021": 0,
    "dropoff_outside_january_2021": 0,
    "negative_fare_amount": 0,
    "zero_or_negative_fare_amount": 0,
    "negative_trip_distance": 0,
    "zero_trip_distance": 0,
    "zero_passenger_count": 0,
    "duplicate_rows_within_each_chunk": 0,
}

for chunk in pd.read_csv(
    DATA_PATH,
    chunksize=CHUNK_SIZE,
    low_memory=False
):
    pickup = pd.to_datetime(
        chunk["tpep_pickup_datetime"], errors="coerce"
    )
    dropoff = pd.to_datetime(
        chunk["tpep_dropoff_datetime"], errors="coerce"
    )

    issue_counts["pickup_outside_january_2021"] += int(
        (~pickup.between("2021-01-01", "2021-01-31 23:59:59")).sum()
    )
    issue_counts["dropoff_outside_january_2021"] += int(
        (~dropoff.between("2021-01-01", "2021-01-31 23:59:59")).sum()
    )

    fare = pd.to_numeric(chunk["fare_amount"], errors="coerce")
    distance = pd.to_numeric(chunk["trip_distance"], errors="coerce")
    passengers = pd.to_numeric(
        chunk["passenger_count"], errors="coerce"
    )

    issue_counts["negative_fare_amount"] += int((fare < 0).sum())
    issue_counts["zero_or_negative_fare_amount"] += int(
        (fare <= 0).sum()
    )
    issue_counts["negative_trip_distance"] += int(
        (distance < 0).sum()
    )
    issue_counts["zero_trip_distance"] += int(
        (distance == 0).sum()
    )
    issue_counts["zero_passenger_count"] += int(
        (passengers == 0).sum()
    )

    issue_counts["duplicate_rows_within_each_chunk"] += int(
        chunk.duplicated().sum()
    )

print("Data quality issue counts:")
print(pd.Series(issue_counts).to_string())


Data quality issue counts:
pickup_outside_january_2021            24
dropoff_outside_january_2021          137
negative_fare_amount                 6769
zero_or_negative_fare_amount         7411
negative_trip_distance                  0
zero_trip_distance                  19952
zero_passenger_count                26726
duplicate_rows_within_each_chunk        0


In [6]:
import platform
import sys

print(platform.system())
print(sys.executable)

Windows
c:\Users\mohamed ahmed fathy\AppData\Local\Programs\Python\Python312\python.exe


In [1]:
import platform
import sys

print("Operating system:", platform.system())
print("Python executable:", sys.executable)

Operating system: Linux
Python executable: /usr/local/bin/python3.12


In [2]:
import os
import sys
import platform

print("Operating system:", platform.system())
print("Python executable:", sys.executable)
print("Working directory:", os.getcwd())
print("Files:", os.listdir("/workspace"))

Operating system: Linux
Python executable: /usr/local/bin/python3.12
Working directory: /workspace
Files: ['.ipynb_checkpoints', 'data', 'docker-compose.yml', 'Dockerfile', 'lab3.ipynb', 'lab3.ipynb.ipynb', 'reports', 'requirements.txt', 'yellow_tripdata_2021-01.csv.gz']
